# gdrive_ine_guatemala_ingest


In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "google-api-python-client", "google-auth", "google-auth-httplib2"], check=True)

import os
from pathlib import Path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from pyspark.sql import functions as F
from pyspark.sql.functions import lit

SCOPES = ["https://www.googleapis.com/auth/drive"]


def _service():
    client_id     = dbutils.secrets.get(scope="semis2_scope", key="GDRIVE_CLIENT_ID")
    client_secret = dbutils.secrets.get(scope="semis2_scope", key="GDRIVE_CLIENT_SECRET")
    refresh_token = dbutils.secrets.get(scope="semis2_scope", key="GDRIVE_REFRESH_TOKEN")

    if client_id and client_secret and refresh_token:
        creds = Credentials(
            token=None,
            refresh_token=refresh_token,
            token_uri="https://oauth2.googleapis.com/token",
            client_id=client_id,
            client_secret=client_secret,
            scopes=SCOPES,
        )
        creds.refresh(Request())
    else:
        raise Exception("Error al conectar con gDrive: no hay secrets cargados...")
    return build("drive", "v3", credentials=creds)


def test_connection() -> None:
    about = _service().about().get(fields="user").execute()
    user = about["user"]
    print(f"Connected as: {user['displayName']} ({user['emailAddress']})")


def download_from_drive(file_id: str, destination: str) -> None:
    request = _service().files().get_media(fileId=file_id)
    Path(destination).parent.mkdir(parents=True, exist_ok=True)
    with open(destination, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(f"Downloaded: {file_id} → {destination}")


def get_folder_id_by_name(folder_name: str, parent_id: str = None) -> str | None:
    query = f"mimeType = 'application/vnd.google-apps.folder' and name = '{folder_name}' and trashed = false"
    if parent_id:
        query += f" and '{parent_id}' in parents"
    result = _service().files().list(q=query, fields="files(id, name)", pageSize=1).execute()
    files = result.get("files", [])
    if files:
        return files[0]["id"]
    print(f"No se encontró ninguna carpeta con el nombre: '{folder_name}'")
    return None


def get_file_id_by_name(file_name: str, folder_id: str = None) -> str | None:
    query = f"mimeType != 'application/vnd.google-apps.folder' and name = '{file_name}' and trashed = false"
    if folder_id:
        query += f" and '{folder_id}' in parents"
    result = _service().files().list(q=query, fields="files(id, name)", pageSize=1).execute()
    files = result.get("files", [])
    if files:
        return files[0]["id"]
    print(f"No se encontró ningún archivo con el nombre: '{file_name}'")
    return None


def list_files_in_folder(folder_id: str) -> list:
    query = f"mimeType != 'application/vnd.google-apps.folder' and '{folder_id}' in parents and trashed = false"
    result = _service().files().list(q=query, fields="files(id, name)", pageSize=100).execute()
    return result.get("files", [])


# ── Ejecución ──────────────────────────────────────────────────
import pandas as pd

spark.sql("CREATE SCHEMA IF NOT EXISTS sandbox")
test_connection()

workspace_tmp_dir = "/Workspace/Shared/tmp/ine"
os.makedirs(workspace_tmp_dir, exist_ok=True)

root_folder_id = get_folder_id_by_name("semis2_raw_data")
ine_folder_id  = get_folder_id_by_name("ine", root_folder_id)
ine_files      = [f for f in list_files_in_folder(ine_folder_id) if f["name"].startswith("ine_defunciones_")]

df_total = None

for file in sorted(ine_files, key=lambda f: f["name"]):
    anio       = int(file["name"].split("_")[-1].replace(".csv", ""))
    final_path = f"{workspace_tmp_dir}/{file['name']}"

    download_from_drive(file["id"], final_path)

    # Databricks Connect: Python descarga al FS del cliente, pero spark.read con
    # scheme `file:` busca en el FS del cluster remoto (donde el archivo no está).
    # Se lee con pandas en el cliente y se envía al cluster vía createDataFrame.
    # CSV de ~80-120k filas, todas como string (inferSchema=False equivalente).
    pdf = pd.read_csv(final_path, dtype=str, keep_default_na=False, encoding="utf-8")

    df_anio = (
        spark.createDataFrame(pdf)
        .withColumn("anio", lit(anio))
        .withColumn("_source_file", lit(file["name"]))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    df_total = df_anio if df_total is None else df_total.unionByName(df_anio, allowMissingColumns=True)

df = df_total
display(df)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sandbox.raw_ine")

# ── Diccionario de catálogos INE ───────────────────────────────
# Códigos -> etiquetas; staging lo usa para decodificar los años que vienen
# codificados (los archivos viejos traen nombres, los nuevos traen códigos).
# Se lee sin header (estructura: Valor, Código, Etiqueta con la primera columna
# rellenada solo en la fila inicial de cada bloque).
dicc_file_id = get_file_id_by_name("ine_diccionario_defunciones.csv", ine_folder_id)
if dicc_file_id:
    dicc_path = f"{workspace_tmp_dir}/ine_diccionario_defunciones.csv"
    download_from_drive(dicc_file_id, dicc_path)

    # Databricks Connect: mismo split-brain que los CSV de defunciones -> pandas.
    # El diccionario no tiene header consistente (estructura Valor,Código,Etiqueta
    # con forward-fill); se lee bruto y el staging lo parsea por patrón.
    # pd.read_csv(header=None) nombra columnas 0,1,2; staging busca _c0,_c1,_c2
    # (convención que spark.read.csv sin header generaba). Se renombran antes
    # de createDataFrame para mantener el contrato del schema.
    pdf_dicc = pd.read_csv(
        dicc_path, header=None, dtype=str,
        keep_default_na=False, encoding="utf-8",
    )
    pdf_dicc.columns = [f"_c{i}" for i in range(len(pdf_dicc.columns))]

    df_dicc = (
        spark.createDataFrame(pdf_dicc)
        .withColumn("_source_file", lit("ine_diccionario_defunciones.csv"))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    display(df_dicc)

    df_dicc.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sandbox.raw_ine_diccionario")
else:
    print("No se encontró ine_diccionario_defunciones.csv en gDrive — staging no podrá decodificar.")

# ── Diccionario CIE-10 (código -> descripción en español) ──────
# staging.ine lo usa para poblar cie10_nombre. Se lee sin header
# (estructura: _c0=código, _c1=descripción; las 2 primeras filas son
# metadatos y se filtran en staging con el patrón CIE-10).
cie10_file_id = get_file_id_by_name("ine_diccionario_cie-10.csv", ine_folder_id)
if cie10_file_id:
    cie10_path = f"{workspace_tmp_dir}/ine_diccionario_cie-10.csv"
    download_from_drive(cie10_file_id, cie10_path)

    # Databricks Connect: Python descarga al FS del cliente, pero spark.read con
    # scheme `file:` busca en el FS del cluster remoto (donde el archivo no está).
    # Se lee con pandas en el cliente y se envía al cluster vía createDataFrame.
    # CSV de ~14k filas, 2 columnas (UTF-8, comillas estándar).
    import pandas as pd
    pdf = pd.read_csv(
        cie10_path, header=None, dtype=str,
        keep_default_na=False, encoding="utf-8",
    )
    pdf = pdf.iloc[:, :2]
    pdf.columns = ["_c0", "_c1"]

    df_cie10 = (
        spark.createDataFrame(pdf)
        .withColumn("_source_file", lit("ine_diccionario_cie-10.csv"))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    display(df_cie10)

    df_cie10.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sandbox.raw_ine_diccionario_cie10")
else:
    print("No se encontró ine_diccionario_cie-10.csv en gDrive — cie10_nombre quedará NULL en staging.")
